# Multiple Linear Regression - شرح ثنائي اللغة / Bilingual Walkthrough

هذا الدفتر مرتب خطوة بخطوة لشرح **الانحدار الخطي المتعدد** مع الاستكشاف، المعالجة، التدريب، والتقييم.
This notebook is organized step-by-step to explain **Multiple Linear Regression** including EDA, preprocessing, training, and diagnostics.

## الترتيب / Flow
1. استيراد المكتبات - Import libraries
2. قراءة البيانات واستكشافها - Load and explore data
3. ترميز المتغيرات الفئوية - Encode categorical variables
4. فحص القيم الشاذة والعلاقات - Outlier and relationship checks
5. فحص الارتباط والتعدد الخطي - Correlation and multicollinearity
6. تجهيز `X` و `y` - Prepare features and target
7. تقسيم البيانات والقياس - Split and scale features
8. تدريب النموذج والتنبؤ - Train model and predict
9. التفسير الإحصائي والتشخيص - Statistical interpretation and diagnostics


In [ ]:
# Step 1) استيراد المكتبات / Import libraries
# Multiple Linear Regression

# Importing the libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Step 2) قراءة البيانات / Load dataset
# Importing the dataset
dataset = pd.read_csv('/content/50_Startups.csv')

In [ ]:
# Step 3) استكشاف البيانات بصريا / Visual exploratory analysis
import matplotlib.pyplot as plt
import seaborn as sns

# Extract feature columns dynamically
features = dataset.columns[:-1]  # All columns except the last one (Profit is target)

# Print features to confirm
print("Features in the dataset:", features.tolist())
# List of features

# Scatter plots
for feature in features:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(data=dataset, x=feature, y='Profit')
    plt.title(f'Scatter Plot: {feature} vs Profit')
    plt.xlabel(feature)
    plt.ylabel('Profit')
    plt.show()


In [ ]:
# Step 4) ترميز المتغير الفئوي State / One-hot encode categorical feature
# One-hot encoding the 'State' column
dataset_encoded = pd.get_dummies(dataset, columns=['State'], drop_first=True, dtype=int)

In [ ]:
# عرض البيانات بعد الترميز / Display encoded dataset
dataset_encoded

In [ ]:
# Step 5) فحص القيم الشاذة / Detect outliers using IQR and boxplots
# Identify outliers using IQR for each numerical feature
for col in dataset_encoded.select_dtypes(include=np.number).columns:  # Select numerical columns
    Q1 = dataset_encoded[col].quantile(0.25)
    Q3 = dataset_encoded[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = dataset_encoded[(dataset_encoded[col] < lower_bound) | (dataset_encoded[col] > upper_bound)]
    print(f"Outliers in {col}:\n{outliers}")

    # Visualization for outlier detection
    plt.figure(figsize=(8, 6))
    sns.boxplot(x=dataset_encoded[col])
    plt.title(f"Boxplot of {col} to detect outliers")
    plt.show()

In [ ]:
# Step 6) مقارنة الخصائص مع Profit بعد الترميز / Compare features vs Profit after encoding
features = dataset_encoded.columns.tolist()
features.remove('Profit')  # Remove the target variable from the feature list

# Plot scatter plots for each feature vs Profit
plt.figure(figsize=(12, 8))

for i, feature in enumerate(features):
    plt.subplot((len(features) + 1) // 2, 2, i + 1)
    plt.scatter(dataset_encoded[feature], dataset_encoded['Profit'], alpha=0.7)
    plt.xlabel(feature)
    plt.ylabel('Profit')
    plt.title(f'Scatter Plot: {feature} vs Profit')

plt.tight_layout()
plt.show()

In [ ]:
# عرض البيانات بعد الفحص / Display dataset after outlier check
dataset_encoded

In [ ]:
# Step 7) ترتيب الأعمدة لتوحيد الشكل / Reorder columns for consistent layout
# Define the desired column order
column_order = ['R&D Spend', 'Administration', 'Marketing Spend', 'State_Florida', 'State_New York', 'Profit']

# Reorder the dataset
dataset_encoded = dataset_encoded[column_order]

# Display the new order
print(dataset_encoded.head())


In [ ]:
# Step 8) مصفوفة الارتباط / Correlation heatmap
# Correlation matrix
# Check the updated dataset
dataset_encoded.head()
correlation_matrix = dataset_encoded.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Step 9) فحص التعدد الخطي VIF / Check multicollinearity with VIF
# 3. Check Multicollinearity Using VIF
# Add a constant column for VIF calculation
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import pandas as pd
import numpy as np

# Assuming dataset_encoded contains your data
X = dataset_encoded.drop(columns=['Profit'])  # Independent variables, Assuming 'Profit' is your target variable
X['constant'] = 1  # Add constant for statsmodels

# Ensure all columns are of numeric type
X = X.astype(float)  # Convert all columns to float

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data['Feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("Variance Inflation Factor (VIF):\n", vif_data)

In [ ]:
# Step 10) تجهيز مصفوفة الخصائص X / Prepare feature matrix X
X = dataset_encoded.iloc[:, :-1].values

In [ ]:
# مراجعة X / Inspect X values
X

In [ ]:
# Step 11) تجهيز المتغير الهدف y / Prepare target vector y
y = dataset_encoded.iloc[:, -1].values

In [ ]:
# مراجعة y / Inspect y values
y

In [ ]:
# Step 12) تقسيم البيانات تدريب/اختبار / Train-test split
# Splitting the dataset into the Training set and Test set
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
# Step 13) قياس الخصائص / Feature scaling
#Feature Scaling
from sklearn.preprocessing import StandardScaler
sc_X = StandardScaler()
X_train = sc_X.fit_transform(X_train)
X_test = sc_X.transform(X_test)

In [ ]:
# Step 14) تدريب نموذج الانحدار الخطي المتعدد / Train multiple linear regression
from sklearn.linear_model import LinearRegression
regressor = LinearRegression()
regressor.fit(X_train, y_train)

In [ ]:
# Step 15) التنبؤ على بيانات الاختبار / Predict on test set
y_pred = regressor.predict(X_test)

In [ ]:
# Step 16) تفسير المعاملات والملخص الإحصائي / Coefficients and statistical summary
import statsmodels.api as sm

# Getting the coefficients and intercept
coefficients = regressor.coef_
intercept = regressor.intercept_

# Calculate p-value using statsmodels
# Adding a constant to X_train for the intercept
X_train_with_constant = sm.add_constant(X_train)

# Fit the statsmodels OLS model to calculate p-values
model = sm.OLS(y_train, X_train_with_constant)
results = model.fit()

# Display the statistical summary
print(results.summary())

In [ ]:
# عرض القيم الحقيقية / Show actual test targets
y_test

In [ ]:
# عرض القيم المتوقعة / Show predicted targets
y_pred

In [ ]:
# Step 17) تحليل البواقي / Residual diagnostics plot
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Get residuals
residuals = y_test - y_pred

# Plot residuals
plt.scatter(y_pred, residuals, color='blue')
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Predicted Profit')
plt.ylabel('Residuals')
plt.title('Residuals vs. Predicted Values')
plt.show()
